# 七零大院：冷面糙汉宠妻成瘾 — v2

**Runtime → Run all**

In [ ]:
!pip install -q requests beautifulsoup4

In [ ]:
import re, requests
from bs4 import BeautifulSoup
from concurrent.futures import ThreadPoolExecutor, as_completed
from google.colab import files

BASE     = 'https://ixdzs8.com/read/609163'
OUTPUT   = '七零大院_冷面糙汉宠妻成瘾.txt'
WORKERS  = 10
MAX_CH   = 484

HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
                  'AppleWebKit/537.36 (KHTML, like Gecko) '
                  'Chrome/124.0.0.0 Safari/537.36',
    'Accept-Language': 'zh-CN,zh;q=0.9',
    'Referer': 'https://ixdzs8.com/',
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8',
    'Accept-Encoding': 'gzip, deflate, br',
    'Connection': 'keep-alive',
    'Cache-Control': 'max-age=0',
    'Upgrade-Insecure-Requests': '1',
}

SKIP = re.compile(r'(ixdzs8|请记住本书首发|www\.|m\.ixdzs|本章未完|点击下一页)', re.I)

CONTENT_SELECTORS = [
    '#page', '#content', '#booktxt', '#BookText', '#htmlContent',
    '#chaptercontent', '#readcontent', '#nr1', '#nr2',
    '.read-content', '.content', '.chapter-content',
    '.novel-content', '.duanzhang', 'article',
]

def count_chinese(text):
    return sum(1 for c in text if '\u4e00' <= c <= '\u9fff')

def extract_best(soup):
    best_el, best_cnt = None, 0
    for sel in CONTENT_SELECTORS:
        el = soup.select_one(sel)
        if el:
            cnt = count_chinese(el.get_text())
            if cnt > best_cnt:
                best_cnt = cnt
                best_el = el
    return best_el, best_cnt

def parse_content(el):
    if not el:
        return ''
    for tag in el.select('script,style,.ad,ins'):
        tag.decompose()
    lines = [p.get_text(' ', strip=True) for p in el.find_all(['p','br'])]
    if not lines:
        lines = [l.strip() for l in el.get_text('\n').splitlines() if l.strip()]
    return '\n'.join(l for l in lines if l and not SKIP.search(l))

# ── Diagnostic: check chapter 1 HTML ─────────────────────────────────────────
print('Checking chapter 1 structure…')
try:
    r = requests.get(f'{BASE}/p1.html', headers=HEADERS, timeout=20)
    r.encoding = 'utf-8'
    soup1 = BeautifulSoup(r.text, 'html.parser')
    print(f'  HTTP status: {r.status_code}')
    print(f'  Page title: {soup1.title.get_text() if soup1.title else "(none)"}')

    print('\n  Selector scores (Chinese char count):')
    for sel in CONTENT_SELECTORS:
        el = soup1.select_one(sel)
        if el:
            cnt = count_chinese(el.get_text())
            preview = el.get_text()[:60].replace('\n',' ')
            print(f'    {sel:25s}  chars={cnt:4d}  preview: {preview}')

    best, cnt = extract_best(soup1)
    print(f'\n  Best content ({cnt} chars): {parse_content(best)[:200]}')

    if cnt < 50:
        print('\n  WARNING: content seems empty — printing raw HTML sample:')
        print(r.text[:3000])
except Exception as e:
    print(f'  ERROR: {e}')

In [ ]:
# ── Download all chapters ─────────────────────────────────────────────────────
def fetch_chapter(n):
    url = f'{BASE}/p{n}.html'
    try:
        r = requests.get(url, headers=HEADERS, timeout=20)
        if r.status_code == 404:
            return n, None, None
        r.encoding = 'utf-8'
        soup = BeautifulSoup(r.text, 'html.parser')

        # Title
        for t_sel in ['h1', '.chapter-title', '.title', 'h2', 'h3']:
            t = soup.select_one(t_sel)
            if t and len(t.get_text(strip=True)) > 1:
                title = t.get_text(strip=True)
                break
        else:
            title = f'第{n}章'

        # Content
        best_el, cnt = extract_best(soup)
        content = parse_content(best_el)
        return n, title, content
    except Exception as e:
        return n, f'第{n}章', f'[error: {e}]'

print(f'Downloading {MAX_CH} chapters with {WORKERS} workers…')
results = {}
with ThreadPoolExecutor(max_workers=WORKERS) as ex:
    futures = {ex.submit(fetch_chapter, n): n for n in range(1, MAX_CH + 1)}
    done = 0
    for f in as_completed(futures):
        n, title, content = f.result()
        if title is not None:
            results[n] = (title, content)
        done += 1
        if done % 50 == 0:
            print(f'  {done}/{MAX_CH} done ({len(results)} valid chapters found)')

chapters = sorted((n, t, c) for n, (t, c) in results.items())
print(f'\nTotal: {len(chapters)} chapters')

# Show sample
if chapters:
    n, t, c = chapters[0]
    print(f'Chapter 1 title: {t}')
    print(f'Chapter 1 content preview: {c[:200] if c else "(empty)"}')

with open(OUTPUT, 'w', encoding='utf-8') as f:
    for n, title, content in chapters:
        f.write(f'\n\n第{n}章　{title}\n\n{content}\n')

print('\n✓ Done! Downloading…')
files.download(OUTPUT)